# Laptop Price Prediction — Feature Engineering


### Objective

The goal of this notebook is to transform the cleaned laptop dataset into meaningful features that can improve the performance of machine learning models.

Feature engineering will focus on creating domain-relevant features from existing laptop specifications while avoiding unnecessary transformations.

### Engineered Features

- Total Storage Capacity
- Total Display Pixels
- Pixel Density (PPI)
- Warranty Information Handling

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("ggplot")
%matplotlib inline

In [ ]:
import os

os.makedirs("../data/processed", exist_ok=True)

In [ ]:
df = pd.read_csv("../data/processed/laptops_cleaned.csv")

df.head()

In [ ]:
df.shape

In [ ]:
df.info()

### Convert `year_of_warranty` to Numeric

The `year_of_warranty` column is converted to numeric format, with invalid values replaced by `NaN`.

In [ ]:
# Already converted to numeric in 02_data_cleaning.ipynb — repeated here as a
# harmless safeguard in case this notebook is ever run on a differently-prepared input
df["year_of_warranty"] = pd.to_numeric(df["year_of_warranty"], errors='coerce')

In [ ]:
df['year_of_warranty'].value_counts(dropna=False)

## Feature Engineering

Feature engineering combines existing variables to create features that represent more meaningful characteristics of a laptop.

The engineered features are based on the physical and technical relationships between laptop specifications.

### 1. Total Storage Capacity

A laptop can contain both primary and secondary storage.

Combining these capacities provides a single measure of the total available storage:

Total Storage = Primary Storage + Secondary Storage

This may help the model capture the overall storage configuration of a laptop.

In [ ]:
df['total_storage_capacity'] = df['primary_storage_capacity']+df['secondary_storage_capacity']

### 2. Total Display Pixels

Display resolution is represented by width and height separately.

Multiplying these values gives the total number of pixels available on the display:

Total Pixels = Resolution Width × Resolution Height

This provides a single measure of display resolution and may capture differences between standard and high-resolution displays.

In [ ]:
df['total_pixels'] = df['resolution_height'] * df['resolution_width']

### 3. Pixel Density (PPI)

Resolution alone does not describe how densely pixels are packed on a physical display.

Pixel density combines display resolution with physical screen size and provides an estimate of pixels per inch (PPI).

A higher PPI indicates a greater concentration of pixels within the display area.

In [ ]:
df['ppi'] = np.sqrt(df['resolution_height'] ** 2 + df['resolution_width'] ** 2) / df['display_size']

## Inspect Engineered Features

The newly created features are inspected to verify their values and ensure that the transformations have been applied correctly.

In [ ]:
engineered_features = [
    "total_storage_capacity",
    "total_pixels",
    "ppi"
]

df[engineered_features].head()

In [ ]:
df[engineered_features].describe().T.round(2)

## Relationship with Price

The correlation between the engineered numerical features and `Price` is examined to understand whether they contain useful information for price prediction.

Correlation is used as an initial diagnostic rather than as the sole criterion for feature selection.

In [ ]:
feature_correlation = pd.concat(
    [df[engineered_features], df["Price"]],
    axis=1
).corr()["Price"].sort_values(ascending=False).drop("Price")

feature_correlation

In [ ]:
fig, axes = plt.subplots(
    ncols=3, 
    nrows=1,
    figsize = (15, 8)
)

for ax, feature in zip(axes, engineered_features):
    sns.scatterplot(
        data = df, 
        x=feature, 
        y= "Price", 
        alpha = 0.5, 
        ax=ax
    )
    
    ax.set_title(f"{feature} vs Price")
    if(feature == 'ppi'):
        ax.set_xlabel(feature.replace("_", " "))
    else :
        ax.set_xlabel(feature.replace("_", " ").title())
    ax.set_ylabel("Price")
    
plt.tight_layout()
plt.show()

## Missing Values After Feature Engineering

The feature engineering process introduced missing values in `year_of_warranty` by converting the `No information` category into `NaN`.

These missing values will not be imputed at this stage. They will be handled later in the machine learning preprocessing pipeline to avoid applying preprocessing decisions before the train-test split.

In [ ]:
df.isnull().sum().sort_values(ascending=False)

## Feature Distribution After Engineering

The distributions of the engineered features are visualized to verify that the newly created features contain reasonable values and to understand their overall distribution.

In [ ]:
fig, axes = plt.subplots(
    nrows=1, 
    ncols=3, 
    figsize = (10, 4)
)

for ax, feature in zip(axes, engineered_features):
    sns.histplot(
        data=df, 
        x=feature, 
        kde=True, 
        ax=ax
    )
    
    ax.set_title(feature.replace("_", " ").title())
    ax.set_xlabel(feature.replace("_", " ").title())
    ax.set_ylabel("Count")
    
plt.tight_layout()
plt.show()

### Observations

- `total_storage_capacity` is concentrated around common configurations such as 512 GB and 1 TB, with a small number of laptops having substantially higher total storage.
- `total_pixels` is heavily concentrated around common display resolutions, while higher-resolution configurations occur less frequently.
- `ppi` is mainly concentrated around the 130–160 PPI range, with fewer laptops having substantially higher pixel density.
- All three engineered features show non-uniform, right-skewed distributions, reflecting the use of standard hardware configurations and a smaller number of premium specifications.

### Save the dataset

In [ ]:
path = "../data/processed/laptops_features.csv"
df.to_csv(path, index= False)

print(f"data saved successfully: {path}")

## Conclusion

Feature engineering was performed using domain knowledge of laptop hardware specifications.

Three numerical features were created: `total_storage_capacity`, `total_pixels`, and `ppi`.

These features provide higher-level representations of storage and display characteristics and will be evaluated during model development to determine their contribution to predictive performance.